# Practice 1 (Part 2): Neural Network Building, Autograd & Training Pipeline

## Model Architecture (MLP vs. FashionCNN), PyTorch Autograd, Loss Functions & Training Loop
This notebook implements the core modeling and optimization lifecycle for Fashion-MNIST classification in PyTorch. It fulfills the second and third requirements of Practice 1 by constructing both a multi-layer perceptron (MLP) baseline and a deep convolutional neural network (FashionCNN), demonstrating the PyTorch Autograd computational graph, training with Adam/SGD, and checkpoint serialization.

## Roadmap Table
| Step | Description | What it does | Import path |
|---|---|---|---|
| 1 | Environment & Seed Setup | Configures dynamic device and seed for reproducibility | [`PYTORCH_FRAMEWORK_RULES.md`](../../../.agents/rules/PYTORCH_FRAMEWORK_RULES.md) |
| 2 | Model Architecture Design | Implements Baseline MLP and Deep Custom `FashionCNN` | `torch.nn.Module` |
| 3 | PyTorch Autograd Demonstration | Demonstrates forward graph computation and `loss.backward()` | `torch.autograd` |
| 4 | Loss Functions & Optimizers | Configures CrossEntropyLoss and Adam/SGD optimizers | `torch.optim` |
| 5 | Training & Validation Loops | Implements full epoch loop with loss and accuracy tracking | [`docs/lab1/PIPELINE_EXPLANATION.md`](../../../docs/lab1/PIPELINE_EXPLANATION.md) |
| 6 | Checkpoint Serialization | Saves and loads model state_dict with `weights_only=True` | `torch.save` |
| 7 | Training Execution & Curves | Runs training or loads pre-trained weights and plots loss curves | `matplotlib.pyplot` |
---

## References
- Task Brief: [`labs/lab1/QUESTION.md`](../QUESTION.md)
- Pipeline Architecture: [`docs/lab1/PIPELINE_EXPLANATION.md`](../../../docs/lab1/PIPELINE_EXPLANATION.md)
- Presentation Script: [`docs/lab1/PRESENTATION.md`](../../../docs/lab1/PRESENTATION.md)
- PyTorch nn Tutorial: [PyTorch Build Model](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)
- Checkpoint Rules: [`agents/rules/LOGGING_CHECKPOINT_RULES.md`](../../../.agents/rules/LOGGING_CHECKPOINT_RULES.md)
- Framework Governance: [`agents/rules/PYTORCH_FRAMEWORK_RULES.md`](../../../.agents/rules/PYTORCH_FRAMEWORK_RULES.md)
- Notebook Standards: [`agents/rules/NOTEBOOK_HEADER_CONVENTION.md`](../../../.agents/rules/NOTEBOOK_HEADER_CONVENTION.md)


## 1. Environment & Reproducibility Setup

Cấu hình thiết bị và khóa ngẫu nhiên theo [`PYTORCH_FRAMEWORK_RULES.md`](../../../.agents/rules/PYTORCH_FRAMEWORK_RULES.md) đảm bảo toàn bộ quá trình khởi tạo trọng số và nạp dữ liệu diễn ra nhất quán.

In [ ]:
import os
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader, random_split

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Active Device: {device}")


## 2. Thiết Kế Kiến Trúc Mô Hình: Baseline MLP vs. Deep FashionCNN

Nhóm xây dựng hai trường phái kiến trúc để so sánh thực nghiệm:
1. **Baseline MLP**: Trải phẳng ảnh thành vector 784 chiều, qua 3 tầng tuyến tính $784 \to 256 \to 128 \to 10$. Có khoảng 235,000 tham số.
2. **Deep FashionCNN**: Gồm 3 khối tích chập chuyên sâu (Conv2d $\to$ BatchNorm $\to$ ReLU $\to$ MaxPool), tầng pooling không gian `AdaptiveAvgPool2d(1)` và đầu phân loại có Dropout 0.5. Có khoảng 150,000 tham số.

In [ ]:
# --- 1. Kiến trúc Baseline MLP ---
class BaselineMLP(nn.Module):
    def __init__(self, in_features=784, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        return self.net(x)

# --- 2. Kiến trúc Deep FashionCNN ---
class FashionCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2) # 28x28 -> 14x14
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2) # 14x14 -> 7x7
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1)) # 7x7 -> 1x1
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

mlp_model = BaselineMLP().to(device)
cnn_model = FashionCNN().to(device)

print(f"Baseline MLP  Parameters : {sum(p.numel() for p in mlp_model.parameters()):,}")
print(f"Deep CNN      Parameters : {sum(p.numel() for p in cnn_model.parameters()):,}")


## 3. Mổ Xẻ Cơ Chế PyTorch Autograd & Đồ Thị Tính Đạo Hàm

Autograd tự động xây dựng đồ thị tính toán có hướng (DAG). Khi gọi `loss.backward()`, PyTorch tự động tính toán đạo hàm theo quy tắc dây chuyền (Chain Rule) và lưu vào thuộc tính `.grad` của từng tham số.

In [ ]:
# Minh họa tính Autograd với một batch giả lập
dummy_inputs = torch.randn(4, 1, 28, 28, device=device)
dummy_labels = torch.tensor([0, 3, 7, 9], device=device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

# Kiểm tra gradient trước khi backward
first_weight = next(cnn_model.parameters())
print("Gradient ban đầu của lớp Conv1:", first_weight.grad)

# Chạy 1 bước tiến và lùi
optimizer.zero_grad()
outputs = cnn_model(dummy_inputs)
loss = criterion(outputs, dummy_labels)
loss.backward()

print("Loss giả lập:", loss.item())
print("Shape gradient của Conv1 sau backward:", first_weight.grad.shape)
print("Chuẩn L2 của gradient Conv1:", first_weight.grad.norm().item())


## 4. Thiết Lập DataLoader & Chuẩn Bị Dữ Liệu Huấn Luyện

Nạp tập dữ liệu Fashion-MNIST và chia tập train (60,000 ảnh) thành 54,000 ảnh huấn luyện và 6,000 ảnh kiểm định (validation split).

In [ ]:
MEAN, STD = (0.2860,), (0.3530,)
train_tf = transforms.Compose([
    transforms.RandomCrop(28, padding=2),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])
val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

full_train = FashionMNIST(root="data", train=True, download=True, transform=train_tf)
train_sub, val_sub = random_split(full_train, [54000, 6000], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_sub, batch_size=64, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_sub, batch_size=64, shuffle=False)
print(f"Kích thước tập Train: {len(train_sub):,} | Validation: {len(val_sub):,}")


## 5. Vòng Lặp Huấn Luyện Chuẩn Mực PyTorch (Training & Validation Loop)

Đóng gói quy trình 5 bước kinh điển của PyTorch thành hàm `train_epoch()` và `validate_epoch()` có theo dõi hàm mất mát và độ chính xác.

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        
    return running_loss / total, correct / total

def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            
    return running_loss / total, correct / total

print("Đã xây dựng thành công hàm train_epoch và validate_epoch!")


## 6. Tuần Tự Hóa & Lưu Checkpoint An Toàn (`weights_only=True`)

Tuân thủ quy chuẩn [`LOGGING_CHECKPOINT_RULES.md`](../../../.agents/rules/LOGGING_CHECKPOINT_RULES.md), checkpoint chỉ lưu `state_dict` thay vì full class object, và luôn nạp an toàn bằng `weights_only=True`.

In [ ]:
checkpoint_path = Path("../main/weights/fashion-normal.pt")
if not checkpoint_path.exists():
    checkpoint_path = Path("fashion_checkpoint_demo.pt")

# Kiểm tra nếu checkpoint đã tồn tại sẵn từ trước
if checkpoint_path.exists():
    print(f"[Checkpoint Found] Phát hiện file trọng số tại: {checkpoint_path}")
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict) and 'model' in ckpt:
        cnn_model.load_state_dict(ckpt['model'])
        print(f"Nạp thành công mô hình đã train (Epoch {ckpt.get('epoch', 'N/A')}, Val Metric {ckpt.get('best_validation', 'N/A'):.4f})")
    else:
        print("Định dạng checkpoint tùy biến, nạp state dict trực tiếp...")
else:
    print("Chưa có checkpoint sẵn có. Sẽ tiến hành demo 2 epochs nhanh...")


## 7. Huấn Luyện Nhanh & Trực Quan Hóa Đường Cong Loss / Accuracy

Chạy thử nghiệm 2 epochs trên mô hình `FashionCNN` hoặc trực quan hóa kết quả đã huấn luyện để kiểm chứng đồ thị học tập.

In [ ]:
EPOCHS_DEMO = 2
train_losses, train_accs = [], []
val_losses, val_accs = [], []

print(f"Bắt đầu thực nghiệm huấn luyện {EPOCHS_DEMO} epochs trên thiết bị {device}...")
for ep in range(1, EPOCHS_DEMO + 1):
    t_loss, t_acc = train_epoch(cnn_model, train_loader, criterion, optimizer, device)
    v_loss, v_acc = validate_epoch(cnn_model, val_loader, criterion, device)
    train_losses.append(t_loss); train_accs.append(t_acc)
    val_losses.append(v_loss); val_accs.append(v_acc)
    print(f"Epoch {ep:02d} | Train Loss: {t_loss:.4f}, Acc: {t_acc*100:.2f}% | Val Loss: {v_loss:.4f}, Acc: {v_acc*100:.2f}%")

# Trực quan hóa đường cong
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(train_losses, label="Train Loss", color="blue")
ax[0].plot(val_losses, label="Val Loss", color="red")
ax[0].set_title("Đường cong Mất mát (Loss Curve)")
ax[0].set_xlabel("Epochs"); ax[0].legend()

ax[1].plot(train_accs, label="Train Acc", color="blue")
ax[1].plot(val_accs, label="Val Acc", color="green")
ax[1].set_title("Đường cong Độ chính xác (Accuracy Curve)")
ax[1].set_xlabel("Epochs"); ax[1].legend()
plt.tight_layout()
plt.show()
